# Data Analysis & Modeling: Which Submissions Should Enter Human Review?

**Original mock inspired by collected interview topics, not a reported Roblox coding question.** All data are synthetic. Format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. Explain your assumptions and choices while coding.

## Interviewer prompt

You work on a platform's content-review team. At submission time, a model should decide which submissions to send to a human reviewer. This is a queueing decision, not an automatic deletion or account ban.

**Problem 1 — Data analysis (14 minutes).** Load the submission and creator tables, inspect data quality, clean fixed validity errors, and build one row per submission. How common is the positive class, and does its observed rate vary by content type?

**Problem 2 — Modeling (21 minutes).** Train a simple binary classifier and a constant baseline. The team's practice requirement is to catch **at least 80% of abusive submissions on validation**, while making the review queue as small as possible among the model's available score thresholds. How would you select a threshold and evaluate it honestly? Report final test ranking metrics, precision, recall and the fraction of submissions entering review.

Explain your assumptions and choices out loud. Use a train/validation/test split; do not use test results to change the threshold. The 80% requirement is an exercise assumption, not a Roblox policy.

**What this mock exercises:** familiar cleaning, many-to-one joins, binary LogisticRegression and probability metrics; a separate validation set; **precision_recall_curve**; turning probabilities into a review decision.

This dataset is not assumed to have an extremely rare positive class: measure its actual prevalence. A useful classifier and a useful operating threshold are different things.

| Coding time | Focus |
| --- | --- |
| 0–4 min | Load the inputs and define a positive example |
| 4–8 min | Clean without deleting useful examples |
| 8–14 min | Join metadata and analyze the queue population |
| 14–18 min | Separate training, threshold selection and final evaluation |
| 18–25 min | Fit a classifier and a constant baseline |
| 25–31 min | Choose the review threshold on validation |
| 31–35 min | Evaluate the frozen policy and hand it off |

Suggested checkpoints only; they are not official grading criteria. The two problems are data analysis (Steps 1–3) and modeling (Steps 4–7).

**Reference version:** complete code, checks, saved outputs and expandable follow-up answers. Try the practice notebook first.

## Data contract

The files are already supplied. Do not generate data or modify the source CSVs.

**submissions.csv — one logged submission per row before cleaning.**

| Column | Meaning |
| --- | --- |
| submission_id | Event identifier; identical exported rows can repeat |
| creator_id | Key into creators.csv |
| text_length | Number of characters; negative sentinel means unknown |
| link_count | Number of links |
| uppercase_ratio | Uppercase share, between 0 and 1 |
| repeated_token_ratio | Repetition share, between 0 and 1 |
| content_type | chat, description or comment; can be missing |
| label | Historical completed review: clean, spam or abuse |
| review_minutes | Time spent reviewing, available only after submission; not an input |

**creators.csv — one row per creator:** creator_id and account_age_days, measured at submission time; account age may be missing.

**Target for this exercise:** needs_review = 1 exactly when label is **abuse**, otherwise 0. Treating spam as negative is only a simplified abuse-specific queue definition, not a real moderation policy. Assume every historical row received a completed review, so the labels are available for training without selecting only previously flagged submissions.

All listed predictor fields are available at submission time. Exclude IDs, label, needs_review and review_minutes from model inputs. Each creator has one distinct submission in this fixture, so disjoint submission splits are also creator-disjoint. A future dataset with repeated creators would need a split matched to the deployment goal.

Negative text lengths become missing; keep their rows. Unknown predictors are not zero and do not make a valid label unusable. Fit imputers, scalers and category vocabularies on training data only.


## Setup

Use the workspace root as the working directory and select **.venv-codesignal/bin/python**. Inputs are in **data/codesignal_roblox_softmax/**. No downloads, data generation, GPU or new dependencies are needed.

Imports and the data path are supplied below.

Do not edit the source CSVs. These exercises deliberately reuse existing files to focus on new decisions rather than a new dataset generator.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn import metrics

SEED = 42
DATA_DIR = Path("data/codesignal_roblox_softmax")
assert DATA_DIR.is_dir(), "Open the notebook from the workspace root."


## Problem 1 · Step 1: Load the inputs and define a positive example (0–4 min)

Load **submissions** and **creators**. What does each row represent? Inspect shapes, types, missing values, label counts, exact duplicate rows and invalid text lengths. Confirm which fields are known when a new submission arrives.

**Say before coding:** "I will define the positive class and prediction time before choosing a model or dropping rows."

**Possible interviewer questions**

1. What is a positive example in this task?
2. Does having a label in the training file make it a valid feature?
3. What would change if only previously flagged submissions were reviewed?


In [2]:
submissions = pd.read_csv(DATA_DIR / "submissions.csv")
creators = pd.read_csv(DATA_DIR / "creators.csv")
submissions.info()
display(submissions.head(), creators.head())
display(submissions.isna().sum(), creators.isna().sum())
display(submissions["label"].value_counts())
print("exact duplicate rows:", submissions.duplicated().sum())
print("negative text lengths:", submissions["text_length"].lt(0).sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   submission_id         1215 non-null   int64  
 1   creator_id            1215 non-null   int64  
 2   text_length           1215 non-null   float64
 3   link_count            1215 non-null   int64  
 4   uppercase_ratio       1215 non-null   float64
 5   repeated_token_ratio  1215 non-null   float64
 6   content_type          1175 non-null   object 
 7   label                 1215 non-null   object 
 8   review_minutes        1215 non-null   float64
dtypes: float64(4), int64(3), object(2)
memory usage: 85.6+ KB


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918
1,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087
2,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863
3,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839
4,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389


,creator_id,account_age_days
0,0,1395.0
1,1,614.0
2,2,1387.0
3,3,124.0
4,4,480.0


submission_id            0
creator_id               0
text_length              0
link_count               0
uppercase_ratio          0
repeated_token_ratio     0
content_type            40
label                    0
review_minutes           0
dtype: int64

creator_id           0
account_age_days    45
dtype: int64

label
clean    713
abuse    286
spam     216
Name: count, dtype: int64

exact duplicate rows: 15
negative text lengths: 35


In [3]:
assert creators["creator_id"].is_unique
assert submissions["label"].notna().all()
assert set(submissions["label"]) == {"clean", "spam", "abuse"}
assert submissions["creator_id"].isin(creators["creator_id"]).all()


<details>
<summary>Follow-up answers</summary>

**What is a positive example in this task?**

A submission whose completed historical label is abuse. The model will estimate that probability at submission time.

**Does having a label in the training file make it a valid feature?**

No. A historical outcome is known offline but not when the live decision is made.

**What would change if only previously flagged submissions were reviewed?**

The labels would be selectively observed. Offline evaluation on that sample would not directly estimate performance on all submissions; collect a suitable random audit sample or address selection bias.

</details>


## Problem 1 · Step 2: Clean without deleting useful examples (4–8 min)

Create **clean** without changing submissions. Remove identical exported rows. Check that submission_id is now unique; stop and investigate if it is not. Replace invalid negative text_length values with missing values, keeping those submissions. Do not learn imputation values yet.

**Say before coding:** "I will separate fixed validity rules from preprocessing statistics that must be fitted on training data."

**Possible interviewer questions**

1. Why not drop every row with a missing field?
2. Why is missing text length different from a length of zero?
3. Can removing exact duplicate rows guarantee unique event IDs?


In [4]:
clean = submissions.drop_duplicates().copy()
assert clean["submission_id"].is_unique, "Investigate conflicting event records."
clean["text_length"] = clean["text_length"].mask(clean["text_length"].lt(0))
display(clean.loc[clean["text_length"].isna()].head())


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes
41,41,41,NaN,0,0.058686,0.555537,description,spam,5.580240
85,85,85,NaN,1,0.266793,0.427931,chat,clean,1.407615
87,87,87,NaN,0,0.218950,0.298009,chat,abuse,5.100299
92,92,92,NaN,1,0.092987,0.298898,comment,clean,1.338601
98,98,98,NaN,0,0.092105,0.046086,chat,clean,1.564299


In [5]:
assert clean["submission_id"].is_unique
assert clean["text_length"].dropna().ge(0).all()
assert len(clean) == len(submissions.drop_duplicates())
assert clean["text_length"].isna().sum() >= submissions["text_length"].lt(0).sum() - submissions.duplicated().sum()


<details>
<summary>Follow-up answers</summary>

**Why not drop every row with a missing field?**

A row may still contain a reliable label and several useful features. Blanket deletion wastes data and can change the population.

**Why is missing text length different from a length of zero?**

Missing means unknown. Zero would assert that the text was empty.

**Can removing exact duplicate rows guarantee unique event IDs?**

No. Two rows with the same event ID can disagree in another column. Check the key separately.

</details>


## Problem 1 · Step 3: Join metadata and analyze the queue population (8–14 min)

Create **frame** with one row per submission and creator metadata attached. Define **needs_review** from the historical label. Report its prevalence overall and a **content_summary** with submission counts and abuse rates by content_type, retaining the missing-type group. Explain what these rates can and cannot establish.

**Say before coding:** "I will preserve the submission unit and check that the metadata join does not multiply examples."

**Possible interviewer questions**

1. Why validate this join as many-to-one?
2. Does a higher abuse rate for one content type prove that the type causes abuse?
3. Would accuracy alone tell us whether this review queue is useful?


In [6]:
frame = clean.merge(creators, on="creator_id", how="left", validate="many_to_one")
frame["needs_review"] = frame["label"].eq("abuse").astype(int)
content_summary = frame.groupby("content_type", dropna=False).agg(
    submissions=("submission_id", "size"),
    abuse_rate=("needs_review", "mean"),
)
print("positive fraction:", frame["needs_review"].mean())
display(content_summary)


positive fraction: 0.2375


,submissions,abuse_rate
content_type,,
chat,367,0.256131
comment,411,0.260341
description,382,0.204188
NaN,40,0.150000


In [7]:
assert len(frame) == len(clean)
assert frame["submission_id"].is_unique
np.testing.assert_array_equal(frame["needs_review"], frame["label"].eq("abuse").astype(int))
assert content_summary["submissions"].sum() == len(frame)


<details>
<summary>Follow-up answers</summary>

**Why validate this join as many-to-one?**

Submission rows may share a creator, but the creator table must have one row per key. This prevents an accidental many-to-many expansion.

**Does a higher abuse rate for one content type prove that the type causes abuse?**

No. These are descriptive observational rates. Other user and content differences may explain them.

**Would accuracy alone tell us whether this review queue is useful?**

No. Compare against prevalence and a constant baseline, then inspect ranking quality, missed abuse and review workload.

</details>


## Problem 2 · Step 4: Separate training, threshold selection and final evaluation (14–18 min)

Which columns should be features, and which should be excluded? Create **numeric_features**, **categorical_features**, **X** and **y**. Create disjoint **X_train / y_train**, **X_valid / y_valid** and **X_test / y_test**, approximately 60% / 20% / 20%, preserving class balance. Explain the role of each set.

**Say before coding:** "Training fits the model, validation chooses the threshold, and the untouched test set estimates the final policy."

**Possible interviewer questions**

1. Why does this task need validation as well as test?
2. Why stratify these splits?
3. What if the deployment goal were performance on next month's submissions?


In [8]:
numeric_features = ["text_length", "link_count", "uppercase_ratio",
                    "repeated_token_ratio", "account_age_days"]
categorical_features = ["content_type"]
X = frame[numeric_features + categorical_features]
y = frame["needs_review"]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED
)
print("train / validation / test:", len(X_train), len(X_valid), len(X_test))


train / validation / test: 720 240 240


In [9]:
for left, right in [(X_train, X_valid), (X_train, X_test), (X_valid, X_test)]:
    assert set(left.index).isdisjoint(right.index)
for inputs, target in [(X_train, y_train), (X_valid, y_valid), (X_test, y_test)]:
    assert inputs.index.equals(target.index)
    assert set(target) == {0, 1}
assert len(X_train) + len(X_valid) + len(X_test) == len(frame)
assert not {"label", "needs_review", "review_minutes", "creator_id", "submission_id"} & set(X.columns)


<details>
<summary>Follow-up answers</summary>

**Why does this task need validation as well as test?**

The threshold is a learned decision. Choosing it on test labels would make the final test report optimistic.

**Why stratify these splits?**

Maintain approximately similar class proportions across subsets, especially when the positive class is smaller. It does not solve dependence or time leakage.

**What if the deployment goal were performance on next month's submissions?**

Use a chronological evaluation with features and fully matured labels available at each cutoff. A random split answers a different question.

</details>


## Problem 2 · Step 5: Fit a classifier and a constant baseline (18–25 min)

Fit **model**, a LogisticRegression pipeline with numeric imputation/scaling and categorical imputation/encoding. Fit **baseline**, a DummyClassifier using training class frequencies. Use training data only. What objective is the classifier minimizing, and why is the constant model a useful comparison?

**Say before coding:** "I will start with a small interpretable model and keep every learned preprocessing statistic inside its pipeline."

**Possible interviewer questions**

1. What does LogisticRegression learn?
2. Does class_weight="balanced" remove the need to choose a threshold?
3. What does the prior baseline predict?


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=500, random_state=SEED)),
])
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)


In [11]:
np.testing.assert_array_equal(model.classes_, [0, 1])
learned_medians = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned_medians, X_train[numeric_features].median().to_numpy())
np.testing.assert_allclose(baseline.class_prior_[1], y_train.mean())


<details>
<summary>Follow-up answers</summary>

**What does LogisticRegression learn?**

A weighted sum plus an intercept, converted to a positive-class probability by sigmoid. With default L2 settings it minimizes binary cross-entropy plus a weight penalty.

**Does class_weight="balanced" remove the need to choose a threshold?**

No. It changes the training objective and can change the probability scale. Threshold and probability quality still need validation. We do not turn it on automatically here.

**What does the prior baseline predict?**

The same training positive-class fraction for every row; it has no ability to rank submissions.

</details>


## Problem 2 · Step 6: Choose the review threshold on validation (25–31 min)

Create **valid_prob**. Build a **threshold_table** describing precision and recall at the available validation score thresholds. Choose **threshold** to meet validation recall of at least **recall_target = 0.80**, while sending as few validation submissions as possible for review.

What does lowering the threshold do to queue volume? Report **validation_decision**, the selected precision, recall and review fraction. Use validation only. If the requirement cannot be evaluated or no candidate is feasible, explain that rather than inventing a successful result.

**Say before coding:** "I am choosing an operating point for a stated review objective; this is not a promise that future recall will equal validation recall."

**Possible interviewer questions**

1. How does precision_recall_curve align its outputs?
2. Why choose the largest threshold that still meets the recall requirement?
3. Would you use the same threshold for automatic deletion?


In [12]:
valid_prob = model.predict_proba(X_valid)[:, 1]
precision, recall, thresholds = metrics.precision_recall_curve(y_valid, valid_prob)
threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})
recall_target = 0.80
feasible = threshold_table.loc[threshold_table["recall"].ge(recall_target)]
if feasible.empty:
    raise ValueError("No feasible validation threshold; discuss the requirement.")
threshold = float(feasible["threshold"].max())
validation_decision = (valid_prob >= threshold).astype(int)
display(threshold_table.loc[threshold_table["threshold"].eq(threshold)])
print("validation review fraction:", validation_decision.mean())
print("validation recall:", metrics.recall_score(y_valid, validation_decision))


,threshold,precision,recall
102,0.158531,0.333333,0.807018


validation review fraction: 0.575
validation recall: 0.8070175438596491


In [13]:
assert len(threshold_table) == len(thresholds)
assert len(precision) == len(recall) == len(thresholds) + 1
assert metrics.recall_score(y_valid, validation_decision) >= recall_target
assert threshold == feasible["threshold"].max()
assert ((validation_decision == 0) | (validation_decision == 1)).all()


<details>
<summary>Follow-up answers</summary>

**How does precision_recall_curve align its outputs?**

Precision and recall have one extra terminal entry with no associated threshold. Pair thresholds with precision[:-1] and recall[:-1].

**Why choose the largest threshold that still meets the recall requirement?**

For decisions score >= threshold on these fixed validation scores, a larger threshold selects a subset and cannot increase review volume. Ties are kept together. This minimizes volume among the available score thresholds, not necessarily maximizes precision.

**Would you use the same threshold for automatic deletion?**

Not automatically. The action changes the cost of false positives. State a new precision/cost requirement and validate the corresponding policy.

</details>


## Problem 2 · Step 7: Evaluate the frozen policy and hand it off (31–35 min)

Freeze the fitted model and selected threshold. Create **test_prob**, **test_decision** and **scores** comparing the prior baseline and model on ROC-AUC, AP and log loss. Create **policy_scores** comparing threshold 0.5 and the selected threshold on precision, recall, F1 and review fraction.

Does the validation recall target also hold on test? Report the result honestly without adjusting the policy. Summarize the trade-off and what you would monitor after deployment. Confusion-matrix display and new-submission inference are optional handoff follow-ups below.

**Say before coding:** "The test set evaluates one frozen policy. I will report both abuse coverage and the human-review workload, including any missed target."

**Possible interviewer questions**

1. Why do ROC-AUC and AP receive probabilities rather than 0/1 decisions?
2. Why might test recall fall below 80%?
3. What if there are not enough reviewers for the resulting queue?


In [14]:
test_prob = model.predict_proba(X_test)[:, 1]
test_decision = (test_prob >= threshold).astype(int)
def ranking_scores(probability):
    return {
        "ROC_AUC": metrics.roc_auc_score(y_test, probability),
        "AP": metrics.average_precision_score(y_test, probability),
        "log_loss": metrics.log_loss(y_test, probability, labels=[0, 1]),
    }
scores = pd.DataFrame({
    "prior": ranking_scores(baseline.predict_proba(X_test)[:, 1]),
    "logistic": ranking_scores(test_prob),
}).T
def policy_metrics(decision):
    return {
        "precision": metrics.precision_score(y_test, decision, zero_division=0),
        "recall": metrics.recall_score(y_test, decision, zero_division=0),
        "F1": metrics.f1_score(y_test, decision, zero_division=0),
        "review_fraction": decision.mean(),
    }
policy_scores = pd.DataFrame({
    "threshold_0.5": policy_metrics((test_prob >= 0.5).astype(int)),
    "selected_threshold": policy_metrics(test_decision),
}).T
display(scores, policy_scores)


,ROC_AUC,AP,log_loss
prior,0.500000,0.237500,0.548181
logistic,0.722366,0.505524,0.483278


,precision,recall,F1,review_fraction
threshold_0.5,0.565217,0.228070,0.325000,0.095833
selected_threshold,0.317241,0.807018,0.455446,0.604167


In [15]:
assert test_prob.shape == test_decision.shape == (len(y_test),)
assert np.isfinite(test_prob).all()
np.testing.assert_allclose(scores.loc["prior", "AP"], y_test.mean())
np.testing.assert_allclose(scores.loc["prior", "ROC_AUC"], 0.5)
np.testing.assert_allclose(policy_scores.loc["selected_threshold", "review_fraction"], test_decision.mean())


<details>
<summary>Follow-up answers</summary>

**Why do ROC-AUC and AP receive probabilities rather than 0/1 decisions?**

They evaluate ranking over possible thresholds. A single binary decision discards most score ordering information.

**Why might test recall fall below 80%?**

Finite samples, a different population or drift can move the operating point. The validation requirement is not a guarantee; do not retune on the final test set.

**What if there are not enough reviewers for the resulting queue?**

The recall and capacity requirements may conflict. Quantify the trade-off on validation, discuss prioritization or added capacity, and do not silently claim both are satisfied.

</details>


## Optional follow-ups — outside the 35-minute core

**Handoff follow-up:** Show a confusion matrix. Score one new_submission with a missing numeric field and an unseen content_type, creating new_probability and send_to_review with the frozen model and threshold. Do not fit again.

**Capacity follow-up:** A reviewer can inspect only 10% of submissions. Is that compatible with your recall goal? Use validation to inspect both constraints before touching test. Discuss why a fixed score threshold does not guarantee a fixed review count after drift.

**New-data follow-up:** The abuse fraction changes next month. What would you monitor, and why might precision move even if the model's conditional detection rates did not change?

**API notes, if needed:** precision_recall_curve produces arrays across score thresholds; unlike ROC-AUC/AP, precision_score and recall_score expect the chosen class decisions.


In [16]:
display(pd.DataFrame(metrics.confusion_matrix(y_test, test_decision, labels=[0, 1]),
    index=["actual: not abuse", "actual: abuse"],
    columns=["not queued", "queued"]))
new_submission = pd.DataFrame([{
    "text_length": np.nan, "link_count": 1, "uppercase_ratio": 0.6,
    "repeated_token_ratio": 0.1, "account_age_days": 30,
    "content_type": "NEW_TYPE",
}])
new_probability = model.predict_proba(new_submission)[:, 1]
send_to_review = new_probability >= threshold
print("new probability / review decision:", new_probability[0], send_to_review[0])
assert new_probability.shape == send_to_review.shape == (1,)

capacity = 0.10
capacity_table = threshold_table.copy()
capacity_table["review_fraction"] = [
    (valid_prob >= value).mean() for value in capacity_table["threshold"]
]
feasible_both = capacity_table.loc[
    capacity_table["recall"].ge(recall_target)
    & capacity_table["review_fraction"].le(capacity)
]
print("thresholds satisfying both validation constraints:", len(feasible_both))
if feasible_both.empty:
    print("No available threshold satisfies both; discuss the trade-off.")
else:
    display(feasible_both.tail())


,not queued,queued
actual: not abuse,84,99
actual: abuse,11,46


new probability / review decision: 0.7690456283170747 True
thresholds satisfying both validation constraints: 0
No available threshold satisfies both; discuss the trade-off.


## Scope and sources

The user-supplied **Data Analysis & Modeling Coding Interview** guide supplies the format: data loading, analysis, feature engineering, a baseline model, evaluation and live explanation in a CodeSignal notebook. It does not specify these exact tasks or APIs.

The local interview collection separates reported **Question/Asked** topics from our explanatory **Answer** material. This mock adapts the topics for practice; it is not evidence that Roblox used this coding problem. Synthetic rates and thresholds are not real product statistics or policy.

The collected report topics motivating this original mock are harmful-content detection ([1125288](https://www.1point3acres.com/bbs/thread-1125288-1-1.html)), classification loss ([1147380](https://www.1point3acres.com/bbs/thread-1147380-1-1.html)), and evaluation ([1161498](https://www.1point3acres.com/bbs/thread-1161498-1-1.html)). The source index is [08_company_interviews.md](08_company_interviews.md). Those reports do **not** establish this exact notebook task, the 80% requirement or these CSVs as actual interview material.

Implementation reference: [sklearn precision_recall_curve](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.precision_recall_curve.html).
